# Make a synthetic twin of any dataset (no LLM, no API key)

You have a dataset you can't share: customer records, patient rows, anything with names and money in it. You still need data that behaves like it for demos, tests, tutorials, and model prototyping.

This notebook shows two things, both running entirely inside this kernel:

1. **`misata.mimic()`**: point it at a CSV and get a synthetic twin that matches the distributions but contains none of the original rows.
2. **`misata.generate()`**: skip the source data entirely. Describe the dataset you want, including the numbers it must hit, and a deterministic math engine builds it.

[Misata](https://github.com/rasinmuhammed/misata) is an open-source Python library (MIT). The generation engine is pure math: seeded, vectorized NumPy. No model call, no tokens, nothing leaves the kernel. The mechanism behind its exact-outcome guarantee is formalized in [this arXiv paper](https://arxiv.org/abs/2606.08736).

In [ ]:
%pip install -q misata
import warnings; warnings.filterwarnings("ignore")
import misata
print("misata", misata.__version__)

## Part 1: the synthetic twin

We'll use the Titanic dataset because everyone on Kaggle knows its shape. Swap in any CSV you like: `misata.mimic()` profiles every column (distribution fit, cardinality, date ranges, semantic type) and generates fresh rows from the profile, never from the data.

In [ ]:
import pandas as pd
from pathlib import Path

# On Kaggle the Titanic competition data lives under /kaggle/input.
kaggle_path = Path("/kaggle/input/titanic/train.csv")
if kaggle_path.exists():
    real = pd.read_csv(kaggle_path)
else:
    real = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

real.head(3)

In [ ]:
twin = misata.mimic(real, rows=2000, seed=42, table_name="passengers")["passengers"]
twin.head(3)

Two thousand passengers who never existed. Same columns, same statistical shape, and `seed=42` means you get the identical twin every time you run this, on any machine.

## Does the twin actually behave like the original?

Looking at three rows proves nothing. Overlay the distributions.

In [ ]:
import matplotlib.pyplot as plt

numeric = [c for c in ["Age", "Fare"] if c in real.columns and c in twin.columns]
fig, axes = plt.subplots(1, len(numeric), figsize=(11, 3.5))
for ax, colname in zip(axes, numeric):
    ax.hist(real[colname].dropna(), bins=30, alpha=0.55, label="real", density=True)
    ax.hist(twin[colname].dropna(), bins=30, alpha=0.55, label="synthetic", density=True)
    ax.set_title(colname)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Categorical balance survives too.
for colname in [c for c in ["Pclass", "Sex", "Embarked"] if c in real.columns]:
    cmp = pd.DataFrame({
        "real": real[colname].value_counts(normalize=True).round(3),
        "synthetic": twin[colname].value_counts(normalize=True).round(3),
    })
    print(cmp, "\n")

## And the privacy check

The twin is sampled from fitted distributions, not resampled from rows. No original record can leak into it, which we can simply verify:

In [ ]:
shared = [c for c in real.columns if c in twin.columns]
overlap = pd.merge(real[shared].astype(str), twin[shared].astype(str), how="inner")
print(f"identical rows shared between real and twin: {len(overlap)}")

## Part 2: no source data at all

Mimic needs a CSV to copy the shape of. The more interesting case is when there is no CSV. Describe the dataset, declare the numbers it has to hit, and the engine builds relational tables that satisfy them exactly. Not approximately: the conformance is closed-form, and you can assert it.

In [ ]:
tables = misata.generate(
    "A fintech with 2,000 customers, checking and savings accounts, "
    "and 20,000 transactions including a 2% fraud rate.",
    seed=42,
)

customers = tables["customers"]
accounts = tables["accounts"]
transactions = tables["transactions"]

print({name: len(df) for name, df in tables.items()})
transactions.head(3)

In [ ]:
# The properties you declared hold. Check them yourself.
orphans = (~transactions["account_id"].isin(accounts["account_id"])).sum()
print(f"orphan foreign keys: {orphans}")
print(f"fraud rate: {transactions['is_fraud'].mean():.4f}")

Why this matters for ML work on Kaggle:

- **Class balance on demand.** Need 2% fraud, or 10%, or 50% for a balanced training run? Declare it. No SMOTE, no resampling artifacts.
- **Reproducible by construction.** Same seed, same bytes, in every kernel rerun and on every teammate's machine.
- **Relational, not just tabular.** Parents generate before children and every foreign key resolves, so joins behave like production data.
- **Realism without a model.** Names match genders across cultures, timestamps land where humans put them, categories follow Zipf's law, and review text agrees with its star rating. All deterministic.

## Where to go next

- GitHub (MIT, star it if this was useful): https://github.com/rasinmuhammed/misata
- Docs: https://rasinmuhammed.github.io/misata
- The no-code studio: https://misata.studio
- Using AI agents? Misata ships an MCP server (`pip install "misata[mcp]"`): your agent designs the schema, the engine guarantees the math and returns an integrity proof.
- The math, formalized: https://arxiv.org/abs/2606.08736

If you fork this notebook, try `misata.mimic()` on your own competition's training set and see what the twin gets right.